In [1]:
import os, shutil, imageio_ffmpeg

src = imageio_ffmpeg.get_ffmpeg_exe()
dst_dir = os.path.abspath("../ffmpeg_bin")
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, os.path.join(dst_dir, "ffmpeg.exe"))
os.environ["PATH"] = dst_dir + os.pathsep + os.environ["PATH"]

In [2]:
import subprocess
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout[:50])

ffmpeg version 9.0.2-essentials_build-www.gyan.dev


In [6]:
import time
import torch
from pathlib import Path
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1


asr = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-tiny",
    device = device
)

Loading weights: 100%|██████████| 167/167 [00:00<00:00, 1001.65it/s]


In [ ]:
audio_dir = Path("../data/my_recordings")
files = sorted(audio_dir.glob("*.*"))

print(f"Found {len(files)} audio files in {audio_dir}")
for file in files:
    print(file.name)


Found 6 audio files in ..\data\my_recordings
18_37_wake_up.ogg
5_min_timer.ogg
run_browser.ogg
sound_lower.ogg
sound_upper.ogg
weather.ogg


In [7]:
for file in files:
    start = time.time()
    result = asr(
        str(file), generate_kwargs={"language":"russian","task":"transcribe"}
        
    )
    elapsed = time.time() - start
    print(f"{file.name:25} | {elapsed:4.1f} с | {result['text']}")

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


18_37_wake_up.ogg         |  2.3 с |  Разбудем меня в 1837.
5_min_timer.ogg           |  2.1 с |  Вставь таймер на 5 минут.
run_browser.ogg           |  1.4 с |  Запусти бралвер.
sound_lower.ogg           |  1.6 с |  Сделай звук потюже.
sound_upper.ogg           |  1.7 с |  Делый зуб по громче.
weather.ogg               |  2.2 с |  Какая сегодня порбона.
